# Task 05 - Feature Engineering V1

Bai toan: Tai moi session hien tai cua mot user, dua tren thong tin session hien tai va lich su truoc do cua user, du doan user co phat sinh mua hang trong 30 ngay tiep theo hay khong, va neu co thi tong doanh thu trong 30 ngay tiep theo la bao nhieu.

Notebook nay chi tap trung vao Feature Set V1. Muc tieu quan trong nhat la tao pipeline on dinh cho du lieu lon: moi stage lon deu ghi parquet ra dia de cat DAG Spark, tranh giu qua nhieu lineage trong JVM.

## Pipeline lam viec V1

Nguyen tac chay:

- Chi doc cac cot can cho V1 tu `column_chunks` trong manifest.
- Khong join full clean table neu khong can.
- Moi stage lon ghi parquet vao `_tmp_feature_engineering_30d` roi doc lai neu stage sau can dung.
- Khong goi `toPandas()` tren bang lon.
- Chi dung `session_revenue_signal` va `session_has_purchase_signal` de tao history bang window ket thuc o dong truoc session hien tai.
- Khong dua tin hieu mua/doanh thu cua session hien tai vao feature list.

Cac stage:

1. Stage 1 - Tao modeling base V1: doc key, time, label, current-session V1 raw columns va leakage signal can cho history; ghi `train/test_modeling_base_v1`.
2. Stage 2 - Tach label, feature, leakage: khai bao danh sach label/debug/leakage/feature V1 va validate khong dua leakage vao feature.
3. Stage 3 - Tao current-session features V1 cho muc 4.1-4.3: cast kieu, fill co ban, tao bang current feature.
4. Stage 4 - Tao user-history features V1 cho muc 4.4: window theo `fullVisitorId`, order theo `visit_start_timestamp`, ket thuc tai session truoc.
5. Stage 5 - Join final feature table V1: current + history + label/debug columns, ghi output train/test.
6. Stage 6 - Validation va manifest: row count, key uniqueness, schema, null, leakage, label distribution, ghi `feature_engineering_manifest.json`.

Checkpoint paths:

- `data_pyspark_parquet/_tmp_feature_engineering_30d/train_modeling_base_v1`
- `data_pyspark_parquet/_tmp_feature_engineering_30d/test_modeling_base_v1`
- `data_pyspark_parquet/_tmp_feature_engineering_30d/train_current_session_features_v1`
- `data_pyspark_parquet/_tmp_feature_engineering_30d/test_current_session_features_v1`
- `data_pyspark_parquet/_tmp_feature_engineering_30d/train_user_history_features_v1`
- `data_pyspark_parquet/_tmp_feature_engineering_30d/test_user_history_features_v1`

Final paths:

- `data_pyspark_parquet/train_user_session_features_30d`
- `data_pyspark_parquet/test_user_session_features_30d`
- `data_pyspark_parquet/feature_engineering_manifest.json`

## 0. Setup

In [16]:
import json
import os
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

from pyspark.sql import SparkSession, Window, functions as F
from pyspark.storagelevel import StorageLevel

cwd = Path.cwd().resolve()
if (cwd / "data_pyspark_parquet").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data_pyspark_parquet").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = Path(r"g:/ds")

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

PARQUET_DIR = PROJECT_ROOT / "data_pyspark_parquet"
MANIFEST_PATH = PARQUET_DIR / "clean_30d_label_sessions_manifest.json"

spark = (
    SparkSession.builder
    .appName("week3_task05_feature_engineering_v1")
    .master("local[*]")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

with MANIFEST_PATH.open(encoding="utf-8") as file:
    clean_manifest = json.load(file)

TRAIN_CLEAN_PATH = Path(clean_manifest["train_clean_path"])
TEST_CLEAN_PATH = Path(clean_manifest["test_clean_path"])
JOIN_KEY_COLUMNS = clean_manifest["join_key_columns"]
PARTITION_COLUMNS = clean_manifest["partition_columns"]
CLEAN_COLUMN_CHUNKS = clean_manifest["column_chunks"]

FE_TMP_DIR = PARQUET_DIR / "_tmp_feature_engineering_30d"
TRAIN_MODELING_BASE_V1_PATH = FE_TMP_DIR / "train_modeling_base_v1"
TEST_MODELING_BASE_V1_PATH = FE_TMP_DIR / "test_modeling_base_v1"

TRAIN_FEATURE_OUTPUT_PATH = PARQUET_DIR / "train_user_session_features_30d"
TEST_FEATURE_OUTPUT_PATH = PARQUET_DIR / "test_user_session_features_30d"
FEATURE_ENGINEERING_MANIFEST_PATH = PARQUET_DIR / "feature_engineering_manifest.json"

WRITE_MODE = "overwrite"
MAX_RECORDS_PER_FILE = 500_000

FE_TMP_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Train clean path:", TRAIN_CLEAN_PATH)
print("Test clean path:", TEST_CLEAN_PATH)
print("Feature tmp dir:", FE_TMP_DIR)
print("PySpark Python:", sys.executable)

Project root: G:\ds
Train clean path: g:\ds\data_pyspark_parquet\train_sessions_clean_30d_label
Test clean path: g:\ds\data_pyspark_parquet\test_sessions_clean_30d_label
Feature tmp dir: G:\ds\data_pyspark_parquet\_tmp_feature_engineering_30d
PySpark Python: f:\ide\anaconda\python.exe


## 0.1 Helper doc column chunks

In [17]:
def unique_preserve_order(values):
    seen = set()
    result = []
    for value in values:
        if value not in seen:
            seen.add(value)
            result.append(value)
    return result


def find_chunks_for_requested_columns(requested_columns, chunks=CLEAN_COLUMN_CHUNKS):
    requested_set = set(requested_columns)
    key_partition_set = set(JOIN_KEY_COLUMNS + PARTITION_COLUMNS)
    non_key_requested = requested_set - key_partition_set
    selected_chunks = [chunk for chunk in chunks if non_key_requested.intersection(set(chunk["columns"]))]
    if not selected_chunks and chunks:
        selected_chunks = [chunks[0]]
    return selected_chunks


def read_clean_selected_columns(base_path, requested_columns, chunks=CLEAN_COLUMN_CHUNKS):
    requested_columns = unique_preserve_order(requested_columns)
    selected_chunks = find_chunks_for_requested_columns(requested_columns, chunks)
    result_df = None
    selected_so_far = set()

    for chunk in selected_chunks:
        chunk_path = Path(base_path) / chunk["chunk_id"]
        chunk_available_columns = set(chunk["columns"])
        requested_from_chunk = [
            column for column in requested_columns
            if column in chunk_available_columns and column not in selected_so_far
        ]
        read_columns = unique_preserve_order(JOIN_KEY_COLUMNS + requested_from_chunk)
        one_chunk_df = spark.read.parquet(str(chunk_path)).select(read_columns)

        if result_df is None:
            result_df = one_chunk_df
        else:
            result_df = result_df.join(one_chunk_df, on=JOIN_KEY_COLUMNS, how="left")

        selected_so_far.update(requested_from_chunk)

    missing_columns = [column for column in requested_columns if column not in selected_so_far and column not in JOIN_KEY_COLUMNS]
    if missing_columns:
        raise ValueError(f"Requested columns not found in clean chunks: {missing_columns}")

    final_columns = [column for column in requested_columns if column in result_df.columns]
    return result_df.select(final_columns)


def reset_output_path(output_path):
    output_path = Path(output_path)
    if output_path.exists():
        shutil.rmtree(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

## 0.2 Feature Set V1 contracts

In [18]:
LABEL_COLUMNS = [
    "future_30d_has_purchase",
    "future_30d_revenue",
]

LABEL_DEBUG_COLUMNS = [
    "future_30d_purchase_count",
    "future_30d_first_purchase_timestamp",
    "days_to_first_purchase",
]

LEAKAGE_EXCLUDED_COLUMNS = [
    "session_revenue_signal",
    "session_has_purchase_signal",
    "session_purchase_signal_source",
    "transaction_revenue",
    "total_transaction_revenue",
    "totals_transaction_revenue",
    "totals_total_transaction_revenue",
    "totals_transactions",
]

CURRENT_NUMERIC_TIME_FEATURES_V1 = [
    "visit_number",
    "totals_hits",
    "totals_pageviews",
    "totals_time_on_site",
    "totals_new_visits",
    "is_bounce",
    "session_hour",
    "session_day_of_week",
    "session_month",
]

CURRENT_CATEGORICAL_FEATURES_V1 = [
    "channelGrouping_model",
    "traffic_channel_type_model",
    "device_category_model",
    "geo_country_model",
]

CURRENT_BINARY_FEATURES_V1 = [
    "has_gclid",
    "has_traffic_campaign",
    "has_referral_path",
]

USER_HISTORY_FEATURES_V1 = [
    "user_previous_sessions",
    "user_days_since_first_session",
    "user_days_since_previous_session",
    "user_previous_avg_pageviews",
    "user_previous_avg_time_on_site",
    "user_previous_bounce_rate",
    "user_previous_purchase_count",
    "user_previous_total_revenue",
    "user_days_since_previous_purchase",
]

USER_HISTORY_DERIVED_FLAG_FEATURES_V1 = [
    "is_first_session",
    "has_previous_purchase",
]

FEATURE_SET_V1 = (
    CURRENT_NUMERIC_TIME_FEATURES_V1
    + CURRENT_CATEGORICAL_FEATURES_V1
    + CURRENT_BINARY_FEATURES_V1
    + USER_HISTORY_FEATURES_V1
)

FEATURE_SET_V1_WITH_DERIVED_FLAGS = unique_preserve_order(
    FEATURE_SET_V1 + USER_HISTORY_DERIVED_FLAG_FEATURES_V1
)

print("Feature Set V1 core column count:", len(FEATURE_SET_V1))
print("Feature Set V1 with derived flags column count:", len(FEATURE_SET_V1_WITH_DERIVED_FLAGS))

Feature Set V1 core column count: 25
Feature Set V1 with derived flags column count: 27


## 1. Task 1 - Tao bang nen modeling V1

Bang nen nay la checkpoint dau tien. No gom key, cot thoi gian, label/debug label, cac current-session raw columns can cho V1 va hai leakage signal chi dung de tao history o cac task sau.

Luu y: `session_revenue_signal` va `session_has_purchase_signal` co trong bang nen de tinh previous purchase history, nhung khong duoc dua truc tiep vao feature final.

In [19]:
MODELING_BASE_COLUMNS_V1 = unique_preserve_order(
    JOIN_KEY_COLUMNS
    + [
        "session_date",
        "visit_start_timestamp",
        "session_year",
        "session_month",
        "has_full_30d_label_window",
    ]
    + LABEL_COLUMNS
    + LABEL_DEBUG_COLUMNS
    + CURRENT_NUMERIC_TIME_FEATURES_V1
    + CURRENT_CATEGORICAL_FEATURES_V1
    + CURRENT_BINARY_FEATURES_V1
    + [
        "session_revenue_signal",
        "session_has_purchase_signal",
    ]
)

print("Modeling base V1 columns:", len(MODELING_BASE_COLUMNS_V1))
print(MODELING_BASE_COLUMNS_V1)

Modeling base V1 columns: 29
['fullVisitorId', 'visit_id', 'session_date', 'visit_start_timestamp', 'session_year', 'session_month', 'has_full_30d_label_window', 'future_30d_has_purchase', 'future_30d_revenue', 'future_30d_purchase_count', 'future_30d_first_purchase_timestamp', 'days_to_first_purchase', 'visit_number', 'totals_hits', 'totals_pageviews', 'totals_time_on_site', 'totals_new_visits', 'is_bounce', 'session_hour', 'session_day_of_week', 'channelGrouping_model', 'traffic_channel_type_model', 'device_category_model', 'geo_country_model', 'has_gclid', 'has_traffic_campaign', 'has_referral_path', 'session_revenue_signal', 'session_has_purchase_signal']


In [20]:
def build_modeling_base_v1(input_path, output_path, dataset_name):
    print(f"Building {dataset_name} modeling base V1 from {input_path}")
    base_df = (
        read_clean_selected_columns(input_path, MODELING_BASE_COLUMNS_V1)
        .withColumn("session_date", F.to_date("session_date"))
        .withColumn("visit_start_timestamp", F.col("visit_start_timestamp").cast("timestamp"))
        .withColumn("session_year", F.col("session_year").cast("int"))
        .withColumn("session_month", F.col("session_month").cast("int"))
        .withColumn("has_full_30d_label_window", F.col("has_full_30d_label_window").cast("int"))
        .withColumn("future_30d_has_purchase", F.col("future_30d_has_purchase").cast("int"))
        .withColumn("future_30d_revenue", F.col("future_30d_revenue").cast("double"))
        .withColumn("future_30d_purchase_count", F.col("future_30d_purchase_count").cast("int"))
        .withColumn("session_revenue_signal", F.coalesce(F.col("session_revenue_signal").cast("double"), F.lit(0.0)))
        .withColumn("session_has_purchase_signal", F.coalesce(F.col("session_has_purchase_signal").cast("int"), F.lit(0)))
        .select(MODELING_BASE_COLUMNS_V1)
    )

    reset_output_path(output_path)
    (
        base_df.write
        .mode(WRITE_MODE)
        .option("maxRecordsPerFile", MAX_RECORDS_PER_FILE)
        .partitionBy(*PARTITION_COLUMNS)
        .parquet(str(output_path))
    )

    print(f"Wrote {dataset_name} modeling base V1 to {output_path}")
    return spark.read.parquet(str(output_path))


train_modeling_base_v1_df = build_modeling_base_v1(
    TRAIN_CLEAN_PATH,
    TRAIN_MODELING_BASE_V1_PATH,
    "train",
)

test_modeling_base_v1_df = build_modeling_base_v1(
    TEST_CLEAN_PATH,
    TEST_MODELING_BASE_V1_PATH,
    "test",
)

Building train modeling base V1 from g:\ds\data_pyspark_parquet\train_sessions_clean_30d_label
Wrote train modeling base V1 to G:\ds\data_pyspark_parquet\_tmp_feature_engineering_30d\train_modeling_base_v1
Building test modeling base V1 from g:\ds\data_pyspark_parquet\test_sessions_clean_30d_label
Wrote test modeling base V1 to G:\ds\data_pyspark_parquet\_tmp_feature_engineering_30d\test_modeling_base_v1


## 1.1 Validate bang nen modeling V1

In [21]:
def summarize_modeling_base_v1(df, dataset_name):
    df = df.persist(StorageLevel.DISK_ONLY)
    total_rows = df.count()
    distinct_keys = df.select(*JOIN_KEY_COLUMNS).distinct().count()
    duplicate_key_rows = total_rows - distinct_keys

    row = df.agg(
        F.sum(F.when(F.col("fullVisitorId").isNull() | F.col("visit_id").isNull(), 1).otherwise(0)).alias("null_key_rows"),
        F.sum(F.when(F.col("has_full_30d_label_window").isNull(), 1).otherwise(0)).alias("null_full_window_flag_rows"),
        F.sum(F.when(F.col("has_full_30d_label_window") == 1, 1).otherwise(0)).alias("full_window_rows"),
        F.sum(F.when((F.col("has_full_30d_label_window") == 1) & F.col("future_30d_has_purchase").isNull(), 1).otherwise(0)).alias("null_class_label_full_window_rows"),
        F.sum(F.when((F.col("has_full_30d_label_window") == 1) & F.col("future_30d_revenue").isNull(), 1).otherwise(0)).alias("null_revenue_label_full_window_rows"),
    ).collect()[0].asDict()

    df.unpersist()
    return {
        "dataset": dataset_name,
        "row_count": total_rows,
        "distinct_key_count": distinct_keys,
        "duplicate_key_rows": duplicate_key_rows,
        **row,
    }


train_modeling_base_v1_df = spark.read.parquet(str(TRAIN_MODELING_BASE_V1_PATH))
test_modeling_base_v1_df = spark.read.parquet(str(TEST_MODELING_BASE_V1_PATH))

train_base_summary = summarize_modeling_base_v1(train_modeling_base_v1_df, "train")
test_base_summary = summarize_modeling_base_v1(test_modeling_base_v1_df, "test")

schema_match = train_modeling_base_v1_df.schema == test_modeling_base_v1_df.schema

print("Train modeling base summary:")
print(json.dumps(train_base_summary, indent=2, default=str))
print("Test modeling base summary:")
print(json.dumps(test_base_summary, indent=2, default=str))
print("Train/test schema match:", schema_match)

if not schema_match:
    raise ValueError("Train/test modeling base schema khong khop")
if train_base_summary["duplicate_key_rows"] != 0 or test_base_summary["duplicate_key_rows"] != 0:
    raise ValueError("Bang modeling base co duplicate key fullVisitorId + visit_id")
if train_base_summary["null_full_window_flag_rows"] != 0 or test_base_summary["null_full_window_flag_rows"] != 0:
    raise ValueError("has_full_30d_label_window co null")
if train_base_summary["null_class_label_full_window_rows"] != 0 or test_base_summary["null_class_label_full_window_rows"] != 0:
    raise ValueError("Classification label bi null trong cac dong co full 30d window")
if train_base_summary["null_revenue_label_full_window_rows"] != 0 or test_base_summary["null_revenue_label_full_window_rows"] != 0:
    raise ValueError("Revenue label bi null trong cac dong co full 30d window")

Train modeling base summary:
{
  "dataset": "train",
  "row_count": 1706613,
  "distinct_key_count": 1706613,
  "duplicate_key_rows": 0,
  "null_key_rows": 0,
  "null_full_window_flag_rows": 0,
  "full_window_rows": 1624078,
  "null_class_label_full_window_rows": 0,
  "null_revenue_label_full_window_rows": 0
}
Test modeling base summary:
{
  "dataset": "test",
  "row_count": 401112,
  "distinct_key_count": 401112,
  "duplicate_key_rows": 0,
  "null_key_rows": 0,
  "null_full_window_flag_rows": 0,
  "full_window_rows": 330036,
  "null_class_label_full_window_rows": 0,
  "null_revenue_label_full_window_rows": 0
}
Train/test schema match: True


Sau Task 1, cac task tiep theo se doc lai tu checkpoint `train/test_modeling_base_v1`, khong doc lai nhieu chunks neu khong can. Dieu nay giup cat lineage Spark va giam rui ro JVM bi chet khi notebook chay dai.

## 2. Task 2 - Tach label, feature, leakage

Task nay tao contract ro rang cho modeling V1:

- `LABEL_COLUMNS`: chi dung lam target.
- `LABEL_DEBUG_COLUMNS`: chi dung validation/debug, khong dua vao model.
- `LEAKAGE_EXCLUDED_COLUMNS`: khong dua truc tiep vao model.
- `session_revenue_signal` va `session_has_purchase_signal`: chi duoc dung trong muc 4.4 de tao previous purchase history bang window ket thuc o dong truoc.
- `FEATURE_SET_V1`: danh sach feature V1 core cua muc 4, gom current-session features va user-history features.

Day la stage nhe, chi validate schema/list cot va khong materialize job lon.

In [22]:
KEY_COLUMNS = JOIN_KEY_COLUMNS

TIME_CONTEXT_COLUMNS = [
    "session_date",
    "visit_start_timestamp",
    "session_year",
    "has_full_30d_label_window",
]

HISTORY_SOURCE_SIGNAL_COLUMNS = [
    "session_revenue_signal",
    "session_has_purchase_signal",
]

MANIFEST_LEAKAGE_COLUMNS = clean_manifest.get("label_source_and_leakage_columns", [])
LEAKAGE_EXCLUDED_COLUMNS = unique_preserve_order(LEAKAGE_EXCLUDED_COLUMNS + MANIFEST_LEAKAGE_COLUMNS)

CURRENT_SESSION_FEATURES_V1 = unique_preserve_order(
    CURRENT_NUMERIC_TIME_FEATURES_V1
    + CURRENT_CATEGORICAL_FEATURES_V1
    + CURRENT_BINARY_FEATURES_V1
)

NON_FEATURE_CONTEXT_COLUMNS = unique_preserve_order(
    KEY_COLUMNS
    + TIME_CONTEXT_COLUMNS
    + LABEL_COLUMNS
    + LABEL_DEBUG_COLUMNS
    + HISTORY_SOURCE_SIGNAL_COLUMNS
)

MODEL_EXCLUDED_COLUMNS_V1 = unique_preserve_order(
    KEY_COLUMNS
    + TIME_CONTEXT_COLUMNS
    + LABEL_COLUMNS
    + LABEL_DEBUG_COLUMNS
    + LEAKAGE_EXCLUDED_COLUMNS
)

FEATURE_GROUPS_V1 = {
    "current_numeric_time_features": CURRENT_NUMERIC_TIME_FEATURES_V1,
    "current_categorical_features": CURRENT_CATEGORICAL_FEATURES_V1,
    "current_binary_features": CURRENT_BINARY_FEATURES_V1,
    "user_history_features": USER_HISTORY_FEATURES_V1,
    "user_history_derived_flag_features": USER_HISTORY_DERIVED_FLAG_FEATURES_V1,
}

NULL_FILL_RULES_V1 = {
    "user_previous_sessions": 0,
    "user_previous_total_revenue": 0.0,
    "user_previous_purchase_count": 0,
    "user_previous_avg_pageviews": 0.0,
    "user_previous_avg_time_on_site": 0.0,
    "user_previous_bounce_rate": 0.0,
    "user_days_since_first_session": 0.0,
    "user_days_since_previous_session": -1.0,
    "user_days_since_previous_purchase": -1.0,
    "is_first_session": 1,
    "has_previous_purchase": 0,
}

TRAIN_ONLY_FIT_RULES_V1 = [
    "Feature set V1 is fixed in this notebook before modeling.",
    "No category threshold or bucket threshold is fit in V1 raw feature table.",
    "StringIndexer, OneHotEncoder, imputer learned values, scaler, and model selection will be fit on train only in modeling notebook.",
    "Test data is transformed by the same fixed V1 rules and is not used to choose features or thresholds.",
]

FEATURE_CONTRACT_V1 = {
    "grain": "1 row = 1 fullVisitorId at 1 current session visit_id",
    "join_key_columns": KEY_COLUMNS,
    "time_context_columns": TIME_CONTEXT_COLUMNS,
    "label_columns": LABEL_COLUMNS,
    "label_debug_columns": LABEL_DEBUG_COLUMNS,
    "history_source_signal_columns": HISTORY_SOURCE_SIGNAL_COLUMNS,
    "leakage_excluded_columns": LEAKAGE_EXCLUDED_COLUMNS,
    "model_excluded_columns_v1": MODEL_EXCLUDED_COLUMNS_V1,
    "feature_groups_v1": FEATURE_GROUPS_V1,
    "feature_set_v1": FEATURE_SET_V1,
    "feature_set_v1_with_derived_flags": FEATURE_SET_V1_WITH_DERIVED_FLAGS,
    "null_fill_rules_v1": NULL_FILL_RULES_V1,
    "train_only_fit_rules_v1": TRAIN_ONLY_FIT_RULES_V1,
}

print("Current-session raw features V1:", len(CURRENT_SESSION_FEATURES_V1))
print("User-history features V1:", len(USER_HISTORY_FEATURES_V1))
print("User-history derived flag features V1:", len(USER_HISTORY_DERIVED_FLAG_FEATURES_V1))
print("Final Feature Set V1 core:", len(FEATURE_SET_V1))
print("Final Feature Set V1 with derived flags:", len(FEATURE_SET_V1_WITH_DERIVED_FLAGS))
print("Model excluded columns V1:", len(MODEL_EXCLUDED_COLUMNS_V1))

Current-session raw features V1: 16
User-history features V1: 9
User-history derived flag features V1: 2
Final Feature Set V1 core: 25
Final Feature Set V1 with derived flags: 27
Model excluded columns V1: 19


In [23]:
def find_duplicates(values):
    seen = set()
    duplicates = []
    for value in values:
        if value in seen and value not in duplicates:
            duplicates.append(value)
        seen.add(value)
    return duplicates


def validate_feature_contract_v1(train_columns, test_columns):
    errors = []

    feature_duplicates = find_duplicates(FEATURE_SET_V1)
    if feature_duplicates:
        errors.append(f"FEATURE_SET_V1 co duplicate columns: {feature_duplicates}")

    leakage_overlap = sorted(set(FEATURE_SET_V1).intersection(LEAKAGE_EXCLUDED_COLUMNS))
    if leakage_overlap:
        errors.append(f"FEATURE_SET_V1 bi overlap leakage columns: {leakage_overlap}")

    label_overlap = sorted(set(FEATURE_SET_V1).intersection(LABEL_COLUMNS + LABEL_DEBUG_COLUMNS))
    if label_overlap:
        errors.append(f"FEATURE_SET_V1 bi overlap label/debug label columns: {label_overlap}")

    key_time_overlap = sorted(set(FEATURE_SET_V1).intersection(KEY_COLUMNS + ["session_date", "visit_start_timestamp", "session_year"]))
    if key_time_overlap:
        errors.append(f"FEATURE_SET_V1 bi overlap key/raw time context columns: {key_time_overlap}")

    required_base_columns = unique_preserve_order(NON_FEATURE_CONTEXT_COLUMNS + CURRENT_SESSION_FEATURES_V1)
    missing_train_base = [column for column in required_base_columns if column not in train_columns]
    missing_test_base = [column for column in required_base_columns if column not in test_columns]
    if missing_train_base:
        errors.append(f"Train modeling base thieu cot can cho V1: {missing_train_base}")
    if missing_test_base:
        errors.append(f"Test modeling base thieu cot can cho V1: {missing_test_base}")

    missing_current_features_train = [column for column in CURRENT_SESSION_FEATURES_V1 if column not in train_columns]
    missing_current_features_test = [column for column in CURRENT_SESSION_FEATURES_V1 if column not in test_columns]
    if missing_current_features_train or missing_current_features_test:
        errors.append(
            "Current-session V1 features chua co du trong modeling base: "
            f"train={missing_current_features_train}, test={missing_current_features_test}"
        )

    if errors:
        raise ValueError("\n".join(errors))

    return {
        "feature_set_v1_count": len(FEATURE_SET_V1),
        "feature_set_v1_with_derived_flags_count": len(FEATURE_SET_V1_WITH_DERIVED_FLAGS),
        "current_session_feature_count": len(CURRENT_SESSION_FEATURES_V1),
        "user_history_feature_count": len(USER_HISTORY_FEATURES_V1),
        "user_history_derived_flag_feature_count": len(USER_HISTORY_DERIVED_FLAG_FEATURES_V1),
        "leakage_overlap_count": len(leakage_overlap),
        "label_overlap_count": len(label_overlap),
        "required_base_column_count": len(required_base_columns),
        "status": "pass",
    }


train_modeling_base_v1_df = spark.read.parquet(str(TRAIN_MODELING_BASE_V1_PATH))
test_modeling_base_v1_df = spark.read.parquet(str(TEST_MODELING_BASE_V1_PATH))

feature_contract_validation_v1 = validate_feature_contract_v1(
    train_modeling_base_v1_df.columns,
    test_modeling_base_v1_df.columns,
)

print(json.dumps(feature_contract_validation_v1, indent=2, default=str))

{
  "feature_set_v1_count": 25,
  "feature_set_v1_with_derived_flags_count": 27,
  "current_session_feature_count": 16,
  "user_history_feature_count": 9,
  "user_history_derived_flag_feature_count": 2,
  "leakage_overlap_count": 0,
  "label_overlap_count": 0,
  "required_base_column_count": 29,
  "status": "pass"
}


In [24]:
feature_role_rows_v1 = []

for column in KEY_COLUMNS:
    feature_role_rows_v1.append({"column": column, "role": "key", "feature_group": "key", "model_usage": "exclude"})
for column in TIME_CONTEXT_COLUMNS:
    feature_role_rows_v1.append({"column": column, "role": "time_context", "feature_group": "context", "model_usage": "exclude_or_feature_if_listed"})
for column in LABEL_COLUMNS:
    feature_role_rows_v1.append({"column": column, "role": "label", "feature_group": "target", "model_usage": "target_only"})
for column in LABEL_DEBUG_COLUMNS:
    feature_role_rows_v1.append({"column": column, "role": "label_debug", "feature_group": "target_debug", "model_usage": "debug_only"})
for column in HISTORY_SOURCE_SIGNAL_COLUMNS:
    feature_role_rows_v1.append({"column": column, "role": "history_source_signal", "feature_group": "history_source", "model_usage": "window_history_only"})
for column in CURRENT_NUMERIC_TIME_FEATURES_V1:
    feature_role_rows_v1.append({"column": column, "role": "feature", "feature_group": "current_numeric_time", "model_usage": "feature_v1"})
for column in CURRENT_CATEGORICAL_FEATURES_V1:
    feature_role_rows_v1.append({"column": column, "role": "feature", "feature_group": "current_categorical", "model_usage": "feature_v1"})
for column in CURRENT_BINARY_FEATURES_V1:
    feature_role_rows_v1.append({"column": column, "role": "feature", "feature_group": "current_binary", "model_usage": "feature_v1"})
for column in USER_HISTORY_FEATURES_V1:
    feature_role_rows_v1.append({"column": column, "role": "feature", "feature_group": "user_history", "model_usage": "feature_v1_after_task4"})
for column in USER_HISTORY_DERIVED_FLAG_FEATURES_V1:
    feature_role_rows_v1.append({"column": column, "role": "derived_feature", "feature_group": "user_history_derived_flag", "model_usage": "optional_feature_after_null_fill"})

feature_role_rows_v1_sorted = sorted(
    feature_role_rows_v1,
    key=lambda row: (row["model_usage"], row["feature_group"], row["column"]),
)

for row in feature_role_rows_v1_sorted:
    print(f"{row['column']:<40} {row['role']:<22} {row['feature_group']:<24} {row['model_usage']}")

days_to_first_purchase                   label_debug            target_debug             debug_only
future_30d_first_purchase_timestamp      label_debug            target_debug             debug_only
future_30d_purchase_count                label_debug            target_debug             debug_only
fullVisitorId                            key                    key                      exclude
visit_id                                 key                    key                      exclude
has_full_30d_label_window                time_context           context                  exclude_or_feature_if_listed
session_date                             time_context           context                  exclude_or_feature_if_listed
session_year                             time_context           context                  exclude_or_feature_if_listed
visit_start_timestamp                    time_context           context                  exclude_or_feature_if_listed
has_gclid                         

Sau Task 2, contract V1 da ro: feature final khong overlap label/debug/leakage, va modeling base co du current-session columns cung hai signal can cho history. Task tiep theo se tao `train/test_current_session_features_v1` tu checkpoint modeling base, voi cast/fill nhe va ghi parquet rieng.

## 4.1-4.3 Current-session features trong Feature Set V1

Stage nay doc lai tu checkpoint `train/test_modeling_base_v1`, chi giu key + partition columns + current-session features V1. Cac bien label, debug label va leakage signal khong duoc ghi vao current feature table.

Quy tac fill/cast o V1 la rule co dinh, khong fit tu test:

- Numeric/time null -> 0, rieng `session_hour`, `session_day_of_week`, `session_month` null -> -1.
- Binary/flag null -> 0.
- Categorical null/empty -> `(missing)`.

Output checkpoint:

- `data_pyspark_parquet/_tmp_feature_engineering_30d/train_current_session_features_v1`
- `data_pyspark_parquet/_tmp_feature_engineering_30d/test_current_session_features_v1`

In [25]:
TRAIN_CURRENT_SESSION_FEATURES_V1_PATH = FE_TMP_DIR / "train_current_session_features_v1"
TEST_CURRENT_SESSION_FEATURES_V1_PATH = FE_TMP_DIR / "test_current_session_features_v1"

CURRENT_FEATURE_PARTITION_COLUMNS = PARTITION_COLUMNS
CURRENT_FEATURE_OUTPUT_COLUMNS_V1 = unique_preserve_order(
    KEY_COLUMNS
    + CURRENT_FEATURE_PARTITION_COLUMNS
    + CURRENT_SESSION_FEATURES_V1
)

CURRENT_NUMERIC_FILL_RULES_V1 = {
    "visit_number": 0,
    "totals_hits": 0,
    "totals_pageviews": 0,
    "totals_time_on_site": 0,
    "totals_new_visits": 0,
    "is_bounce": 0,
    "session_hour": -1,
    "session_day_of_week": -1,
    "session_month": -1,
}

CURRENT_CATEGORICAL_FILL_VALUE_V1 = "(missing)"
CURRENT_BINARY_FILL_VALUE_V1 = 0

print("Current feature output columns V1:", len(CURRENT_FEATURE_OUTPUT_COLUMNS_V1))
print(CURRENT_FEATURE_OUTPUT_COLUMNS_V1)

Current feature output columns V1: 19
['fullVisitorId', 'visit_id', 'session_year', 'session_month', 'visit_number', 'totals_hits', 'totals_pageviews', 'totals_time_on_site', 'totals_new_visits', 'is_bounce', 'session_hour', 'session_day_of_week', 'channelGrouping_model', 'traffic_channel_type_model', 'device_category_model', 'geo_country_model', 'has_gclid', 'has_traffic_campaign', 'has_referral_path']


In [26]:
def build_current_session_features_v1(modeling_base_path, output_path, dataset_name):
    print(f"Building {dataset_name} current-session features V1 from {modeling_base_path}")
    base_df = spark.read.parquet(str(modeling_base_path))

    feature_df = base_df.select(CURRENT_FEATURE_OUTPUT_COLUMNS_V1)

    for column, fill_value in CURRENT_NUMERIC_FILL_RULES_V1.items():
        feature_df = feature_df.withColumn(
            column,
            F.coalesce(F.col(column).cast("double"), F.lit(float(fill_value))),
        )

    current_integer_features = [
        "visit_number",
        "totals_hits",
        "totals_pageviews",
        "totals_time_on_site",
        "totals_new_visits",
        "is_bounce",
        "session_hour",
        "session_day_of_week",
        "session_month",
    ]
    for column in current_integer_features:
        feature_df = feature_df.withColumn(column, F.col(column).cast("int"))

    for column in CURRENT_BINARY_FEATURES_V1:
        feature_df = feature_df.withColumn(
            column,
            F.coalesce(F.col(column).cast("int"), F.lit(CURRENT_BINARY_FILL_VALUE_V1)),
        )

    for column in CURRENT_CATEGORICAL_FEATURES_V1:
        feature_df = feature_df.withColumn(
            column,
            F.when(F.trim(F.col(column).cast("string")) == "", F.lit(CURRENT_CATEGORICAL_FILL_VALUE_V1))
            .otherwise(F.coalesce(F.col(column).cast("string"), F.lit(CURRENT_CATEGORICAL_FILL_VALUE_V1))),
        )

    feature_df = feature_df.select(CURRENT_FEATURE_OUTPUT_COLUMNS_V1)

    reset_output_path(output_path)
    (
        feature_df.write
        .mode(WRITE_MODE)
        .option("maxRecordsPerFile", MAX_RECORDS_PER_FILE)
        .partitionBy(*CURRENT_FEATURE_PARTITION_COLUMNS)
        .parquet(str(output_path))
    )

    print(f"Wrote {dataset_name} current-session features V1 to {output_path}")
    return spark.read.parquet(str(output_path))


train_current_session_features_v1_df = build_current_session_features_v1(
    TRAIN_MODELING_BASE_V1_PATH,
    TRAIN_CURRENT_SESSION_FEATURES_V1_PATH,
    "train",
)

test_current_session_features_v1_df = build_current_session_features_v1(
    TEST_MODELING_BASE_V1_PATH,
    TEST_CURRENT_SESSION_FEATURES_V1_PATH,
    "test",
)

Building train current-session features V1 from G:\ds\data_pyspark_parquet\_tmp_feature_engineering_30d\train_modeling_base_v1
Wrote train current-session features V1 to G:\ds\data_pyspark_parquet\_tmp_feature_engineering_30d\train_current_session_features_v1
Building test current-session features V1 from G:\ds\data_pyspark_parquet\_tmp_feature_engineering_30d\test_modeling_base_v1
Wrote test current-session features V1 to G:\ds\data_pyspark_parquet\_tmp_feature_engineering_30d\test_current_session_features_v1


## 4.1-4.3 Validate current-session features V1

In [27]:
def summarize_current_session_features_v1(df, dataset_name):
    df = df.persist(StorageLevel.DISK_ONLY)
    total_rows = df.count()
    distinct_keys = df.select(*KEY_COLUMNS).distinct().count()

    null_count_exprs = [
        F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
        for column in CURRENT_SESSION_FEATURES_V1
    ]
    null_counts = df.agg(*null_count_exprs).collect()[0].asDict()

    binary_validation_columns = unique_preserve_order(CURRENT_BINARY_FEATURES_V1 + ["is_bounce"])
    binary_invalid_exprs = [
        F.sum(F.when(~F.col(column).isin(0, 1), 1).otherwise(0)).alias(column)
        for column in binary_validation_columns
    ]
    binary_invalid_counts = df.agg(*binary_invalid_exprs).collect()[0].asDict()

    category_missing_exprs = [
        F.sum(F.when(F.col(column) == CURRENT_CATEGORICAL_FILL_VALUE_V1, 1).otherwise(0)).alias(column)
        for column in CURRENT_CATEGORICAL_FEATURES_V1
    ]
    category_missing_counts = df.agg(*category_missing_exprs).collect()[0].asDict()

    df.unpersist()
    return {
        "dataset": dataset_name,
        "row_count": total_rows,
        "distinct_key_count": distinct_keys,
        "duplicate_key_rows": total_rows - distinct_keys,
        "feature_null_counts": null_counts,
        "binary_invalid_counts": binary_invalid_counts,
        "category_missing_counts": category_missing_counts,
    }


train_current_session_features_v1_df = spark.read.parquet(str(TRAIN_CURRENT_SESSION_FEATURES_V1_PATH))
test_current_session_features_v1_df = spark.read.parquet(str(TEST_CURRENT_SESSION_FEATURES_V1_PATH))

current_session_schema_match = train_current_session_features_v1_df.schema == test_current_session_features_v1_df.schema
train_current_session_summary_v1 = summarize_current_session_features_v1(train_current_session_features_v1_df, "train")
test_current_session_summary_v1 = summarize_current_session_features_v1(test_current_session_features_v1_df, "test")

print("Train current-session feature summary V1:")
print(json.dumps(train_current_session_summary_v1, indent=2, default=str))
print("Test current-session feature summary V1:")
print(json.dumps(test_current_session_summary_v1, indent=2, default=str))
print("Train/test current-session schema match:", current_session_schema_match)

if not current_session_schema_match:
    raise ValueError("Train/test current-session feature schema khong khop")
if train_current_session_summary_v1["duplicate_key_rows"] != 0 or test_current_session_summary_v1["duplicate_key_rows"] != 0:
    raise ValueError("Current-session feature table co duplicate key fullVisitorId + visit_id")
if any(value != 0 for value in train_current_session_summary_v1["feature_null_counts"].values()):
    raise ValueError("Train current-session feature table van con null")
if any(value != 0 for value in test_current_session_summary_v1["feature_null_counts"].values()):
    raise ValueError("Test current-session feature table van con null")
if any(value != 0 for value in train_current_session_summary_v1["binary_invalid_counts"].values()):
    raise ValueError("Train current-session binary features co gia tri ngoai 0/1")
if any(value != 0 for value in test_current_session_summary_v1["binary_invalid_counts"].values()):
    raise ValueError("Test current-session binary features co gia tri ngoai 0/1")

Train current-session feature summary V1:
{
  "dataset": "train",
  "row_count": 1706613,
  "distinct_key_count": 1706613,
  "duplicate_key_rows": 0,
  "feature_null_counts": {
    "visit_number": 0,
    "totals_hits": 0,
    "totals_pageviews": 0,
    "totals_time_on_site": 0,
    "totals_new_visits": 0,
    "is_bounce": 0,
    "session_hour": 0,
    "session_day_of_week": 0,
    "session_month": 0,
    "channelGrouping_model": 0,
    "traffic_channel_type_model": 0,
    "device_category_model": 0,
    "geo_country_model": 0,
    "has_gclid": 0,
    "has_traffic_campaign": 0,
    "has_referral_path": 0
  },
  "binary_invalid_counts": {
    "has_gclid": 0,
    "has_traffic_campaign": 0,
    "has_referral_path": 0,
    "is_bounce": 0
  },
  "category_missing_counts": {
    "channelGrouping_model": 0,
    "traffic_channel_type_model": 0,
    "device_category_model": 0,
    "geo_country_model": 0
  }
}
Test current-session feature summary V1:
{
  "dataset": "test",
  "row_count": 401112,


Sau muc 4.1-4.3, current-session features V1 da duoc materialize rieng. Muc tiep theo se tao user-history features V1 bang window theo user va bat buoc khong lay session hien tai vao purchase/revenue history.

## 4.4 User-history features trong Feature Set V1

Stage nay tao cac feature lich su theo user. Day la stage co window lon, nen phai ghi checkpoint rieng sau khi tinh xong.

Nguyen tac leakage:

- Window history dung `rowsBetween(Window.unboundedPreceding, -1)`.
- Cac bien mua hang/doanh thu cua session hien tai chi nam trong source de tinh history cho cac session sau.
- Feature `user_previous_purchase_count`, `user_previous_total_revenue`, `user_days_since_previous_purchase` khong bao gom session hien tai.

Output checkpoint:

- `data_pyspark_parquet/_tmp_feature_engineering_30d/train_user_history_features_v1`
- `data_pyspark_parquet/_tmp_feature_engineering_30d/test_user_history_features_v1`

In [28]:
TRAIN_USER_HISTORY_FEATURES_V1_PATH = FE_TMP_DIR / "train_user_history_features_v1"
TEST_USER_HISTORY_FEATURES_V1_PATH = FE_TMP_DIR / "test_user_history_features_v1"

USER_HISTORY_SOURCE_COLUMNS_V1 = unique_preserve_order(
    KEY_COLUMNS
    + PARTITION_COLUMNS
    + [
        "session_date",
        "visit_start_timestamp",
        "totals_pageviews",
        "totals_time_on_site",
        "is_bounce",
        "session_has_purchase_signal",
        "session_revenue_signal",
    ]
)

USER_HISTORY_OUTPUT_COLUMNS_V1 = unique_preserve_order(
    KEY_COLUMNS
    + PARTITION_COLUMNS
    + USER_HISTORY_FEATURES_V1
    + USER_HISTORY_DERIVED_FLAG_FEATURES_V1
)

print("User-history source columns V1:", len(USER_HISTORY_SOURCE_COLUMNS_V1))
print(USER_HISTORY_SOURCE_COLUMNS_V1)
print("User-history output columns V1:", len(USER_HISTORY_OUTPUT_COLUMNS_V1))
print(USER_HISTORY_OUTPUT_COLUMNS_V1)

User-history source columns V1: 11
['fullVisitorId', 'visit_id', 'session_year', 'session_month', 'session_date', 'visit_start_timestamp', 'totals_pageviews', 'totals_time_on_site', 'is_bounce', 'session_has_purchase_signal', 'session_revenue_signal']
User-history output columns V1: 15
['fullVisitorId', 'visit_id', 'session_year', 'session_month', 'user_previous_sessions', 'user_days_since_first_session', 'user_days_since_previous_session', 'user_previous_avg_pageviews', 'user_previous_avg_time_on_site', 'user_previous_bounce_rate', 'user_previous_purchase_count', 'user_previous_total_revenue', 'user_days_since_previous_purchase', 'is_first_session', 'has_previous_purchase']


In [29]:
def build_user_history_features_v1(modeling_base_path, output_path, dataset_name):
    print(f"Building {dataset_name} user-history features V1 from {modeling_base_path}")
    source_df = (
        spark.read.parquet(str(modeling_base_path))
        .select(USER_HISTORY_SOURCE_COLUMNS_V1)
        .withColumn("session_date", F.to_date("session_date"))
        .withColumn("visit_ts_seconds", F.col("visit_start_timestamp").cast("long"))
        .withColumn("pageviews_clean", F.coalesce(F.col("totals_pageviews").cast("double"), F.lit(0.0)))
        .withColumn("time_on_site_clean", F.coalesce(F.col("totals_time_on_site").cast("double"), F.lit(0.0)))
        .withColumn("bounce_clean", F.coalesce(F.col("is_bounce").cast("double"), F.lit(0.0)))
        .withColumn("purchase_signal_clean", F.coalesce(F.col("session_has_purchase_signal").cast("int"), F.lit(0)))
        .withColumn("revenue_signal_clean", F.coalesce(F.col("session_revenue_signal").cast("double"), F.lit(0.0)))
        .repartition("fullVisitorId")
    )

    user_order_window = Window.partitionBy("fullVisitorId").orderBy("visit_ts_seconds", "visit_id")
    user_to_current_window = user_order_window.rowsBetween(Window.unboundedPreceding, Window.currentRow)
    user_previous_window = user_order_window.rowsBetween(Window.unboundedPreceding, -1)

    history_df = (
        source_df
        .withColumn("user_previous_sessions", F.coalesce(F.count(F.lit(1)).over(user_previous_window), F.lit(0)).cast("long"))
        .withColumn("previous_pageviews_sum", F.coalesce(F.sum("pageviews_clean").over(user_previous_window), F.lit(0.0)))
        .withColumn("previous_time_on_site_sum", F.coalesce(F.sum("time_on_site_clean").over(user_previous_window), F.lit(0.0)))
        .withColumn("previous_bounce_sum", F.coalesce(F.sum("bounce_clean").over(user_previous_window), F.lit(0.0)))
        .withColumn("user_previous_purchase_count", F.coalesce(F.sum("purchase_signal_clean").over(user_previous_window), F.lit(0)).cast("long"))
        .withColumn("user_previous_total_revenue", F.coalesce(F.sum("revenue_signal_clean").over(user_previous_window), F.lit(0.0)))
        .withColumn("first_visit_ts_seconds", F.min("visit_ts_seconds").over(user_to_current_window))
        .withColumn("previous_visit_ts_seconds", F.lag("visit_ts_seconds").over(user_order_window))
        .withColumn(
            "previous_purchase_ts_seconds",
            F.max(F.when(F.col("purchase_signal_clean") == 1, F.col("visit_ts_seconds"))).over(user_previous_window),
        )
        .withColumn(
            "user_previous_avg_pageviews",
            F.when(F.col("user_previous_sessions") > 0, F.col("previous_pageviews_sum") / F.col("user_previous_sessions")).otherwise(F.lit(0.0)),
        )
        .withColumn(
            "user_previous_avg_time_on_site",
            F.when(F.col("user_previous_sessions") > 0, F.col("previous_time_on_site_sum") / F.col("user_previous_sessions")).otherwise(F.lit(0.0)),
        )
        .withColumn(
            "user_previous_bounce_rate",
            F.when(F.col("user_previous_sessions") > 0, F.col("previous_bounce_sum") / F.col("user_previous_sessions")).otherwise(F.lit(0.0)),
        )
        .withColumn(
            "user_days_since_first_session",
            F.coalesce((F.col("visit_ts_seconds") - F.col("first_visit_ts_seconds")) / F.lit(86400.0), F.lit(0.0)),
        )
        .withColumn(
            "user_days_since_previous_session",
            F.coalesce((F.col("visit_ts_seconds") - F.col("previous_visit_ts_seconds")) / F.lit(86400.0), F.lit(-1.0)),
        )
        .withColumn(
            "user_days_since_previous_purchase",
            F.coalesce((F.col("visit_ts_seconds") - F.col("previous_purchase_ts_seconds")) / F.lit(86400.0), F.lit(-1.0)),
        )
        .withColumn("is_first_session", F.when(F.col("user_previous_sessions") == 0, F.lit(1)).otherwise(F.lit(0)).cast("int"))
        .withColumn("has_previous_purchase", F.when(F.col("user_previous_purchase_count") > 0, F.lit(1)).otherwise(F.lit(0)).cast("int"))
        .select(USER_HISTORY_OUTPUT_COLUMNS_V1)
    )

    reset_output_path(output_path)
    (
        history_df.write
        .mode(WRITE_MODE)
        .option("maxRecordsPerFile", MAX_RECORDS_PER_FILE)
        .partitionBy(*PARTITION_COLUMNS)
        .parquet(str(output_path))
    )

    print(f"Wrote {dataset_name} user-history features V1 to {output_path}")
    return spark.read.parquet(str(output_path))


train_user_history_features_v1_df = build_user_history_features_v1(
    TRAIN_MODELING_BASE_V1_PATH,
    TRAIN_USER_HISTORY_FEATURES_V1_PATH,
    "train",
)

test_user_history_features_v1_df = build_user_history_features_v1(
    TEST_MODELING_BASE_V1_PATH,
    TEST_USER_HISTORY_FEATURES_V1_PATH,
    "test",
)

Building train user-history features V1 from G:\ds\data_pyspark_parquet\_tmp_feature_engineering_30d\train_modeling_base_v1
Wrote train user-history features V1 to G:\ds\data_pyspark_parquet\_tmp_feature_engineering_30d\train_user_history_features_v1
Building test user-history features V1 from G:\ds\data_pyspark_parquet\_tmp_feature_engineering_30d\test_modeling_base_v1
Wrote test user-history features V1 to G:\ds\data_pyspark_parquet\_tmp_feature_engineering_30d\test_user_history_features_v1


## 4.4.1 Validate user-history features V1

In [30]:
def summarize_user_history_features_v1(df, dataset_name):
    df = df.persist(StorageLevel.DISK_ONLY)
    total_rows = df.count()
    distinct_keys = df.select(*KEY_COLUMNS).distinct().count()

    null_count_exprs = [
        F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
        for column in USER_HISTORY_FEATURES_V1 + USER_HISTORY_DERIVED_FLAG_FEATURES_V1
    ]
    null_counts = df.agg(*null_count_exprs).collect()[0].asDict()

    rule_row = df.agg(
        F.sum(F.when(F.col("user_previous_sessions") < 0, 1).otherwise(0)).alias("negative_previous_sessions"),
        F.sum(F.when((F.col("is_first_session") == 1) & (F.col("user_previous_sessions") != 0), 1).otherwise(0)).alias("first_session_with_previous_sessions"),
        F.sum(F.when((F.col("has_previous_purchase") == 1) & (F.col("user_previous_purchase_count") <= 0), 1).otherwise(0)).alias("has_previous_purchase_without_count"),
        F.sum(F.when((F.col("user_previous_purchase_count") == 0) & (F.col("user_days_since_previous_purchase") != -1), 1).otherwise(0)).alias("no_purchase_but_days_since_purchase_not_minus_one"),
        F.sum(F.when(~F.col("is_first_session").isin(0, 1), 1).otherwise(0)).alias("invalid_is_first_session"),
        F.sum(F.when(~F.col("has_previous_purchase").isin(0, 1), 1).otherwise(0)).alias("invalid_has_previous_purchase"),
    ).collect()[0].asDict()

    df.unpersist()
    return {
        "dataset": dataset_name,
        "row_count": total_rows,
        "distinct_key_count": distinct_keys,
        "duplicate_key_rows": total_rows - distinct_keys,
        "feature_null_counts": null_counts,
        "rule_validation": rule_row,
    }


train_user_history_features_v1_df = spark.read.parquet(str(TRAIN_USER_HISTORY_FEATURES_V1_PATH))
test_user_history_features_v1_df = spark.read.parquet(str(TEST_USER_HISTORY_FEATURES_V1_PATH))

user_history_schema_match = train_user_history_features_v1_df.schema == test_user_history_features_v1_df.schema
train_user_history_summary_v1 = summarize_user_history_features_v1(train_user_history_features_v1_df, "train")
test_user_history_summary_v1 = summarize_user_history_features_v1(test_user_history_features_v1_df, "test")

print("Train user-history feature summary V1:")
print(json.dumps(train_user_history_summary_v1, indent=2, default=str))
print("Test user-history feature summary V1:")
print(json.dumps(test_user_history_summary_v1, indent=2, default=str))
print("Train/test user-history schema match:", user_history_schema_match)

if not user_history_schema_match:
    raise ValueError("Train/test user-history feature schema khong khop")
if train_user_history_summary_v1["duplicate_key_rows"] != 0 or test_user_history_summary_v1["duplicate_key_rows"] != 0:
    raise ValueError("User-history feature table co duplicate key fullVisitorId + visit_id")
if any(value != 0 for value in train_user_history_summary_v1["feature_null_counts"].values()):
    raise ValueError("Train user-history feature table van con null")
if any(value != 0 for value in test_user_history_summary_v1["feature_null_counts"].values()):
    raise ValueError("Test user-history feature table van con null")
if any(value != 0 for value in train_user_history_summary_v1["rule_validation"].values()):
    raise ValueError("Train user-history rule validation failed")
if any(value != 0 for value in test_user_history_summary_v1["rule_validation"].values()):
    raise ValueError("Test user-history rule validation failed")

Train user-history feature summary V1:
{
  "dataset": "train",
  "row_count": 1706613,
  "distinct_key_count": 1706613,
  "duplicate_key_rows": 0,
  "feature_null_counts": {
    "user_previous_sessions": 0,
    "user_days_since_first_session": 0,
    "user_days_since_previous_session": 0,
    "user_previous_avg_pageviews": 0,
    "user_previous_avg_time_on_site": 0,
    "user_previous_bounce_rate": 0,
    "user_previous_purchase_count": 0,
    "user_previous_total_revenue": 0,
    "user_days_since_previous_purchase": 0,
    "is_first_session": 0,
    "has_previous_purchase": 0
  },
  "rule_validation": {
    "negative_previous_sessions": 0,
    "first_session_with_previous_sessions": 0,
    "has_previous_purchase_without_count": 0,
    "no_purchase_but_days_since_purchase_not_minus_one": 0,
    "invalid_is_first_session": 0,
    "invalid_has_previous_purchase": 0
  }
}
Test user-history feature summary V1:
{
  "dataset": "test",
  "row_count": 401112,
  "distinct_key_count": 401112,
  

## 4.5 Feature Set V1 cuoi cung

Feature Set V1 core dung theo thiet ke trong task:

- Numeric/time: 9 features.
- Categorical: 4 features.
- Sparse/binary flags: 3 features.
- User-history: 9 features.

Tong core feature = 25. Hai cot `is_first_session` va `has_previous_purchase` duoc tao them tu rule fill/history va duoc ghi rieng vao `FEATURE_SET_V1_WITH_DERIVED_FLAGS` neu muon dung trong modeling sau nay.

In [31]:
feature_set_v1_summary = {
    "numeric_time_features": CURRENT_NUMERIC_TIME_FEATURES_V1,
    "categorical_features": CURRENT_CATEGORICAL_FEATURES_V1,
    "binary_features": CURRENT_BINARY_FEATURES_V1,
    "user_history_features": USER_HISTORY_FEATURES_V1,
    "derived_flag_features_optional": USER_HISTORY_DERIVED_FLAG_FEATURES_V1,
    "feature_set_v1_core": FEATURE_SET_V1,
    "feature_set_v1_with_derived_flags": FEATURE_SET_V1_WITH_DERIVED_FLAGS,
}

expected_feature_set_v1_count = 25
if len(FEATURE_SET_V1) != expected_feature_set_v1_count:
    raise ValueError(f"FEATURE_SET_V1 phai co {expected_feature_set_v1_count} cot, hien co {len(FEATURE_SET_V1)}")

print(json.dumps({
    "numeric_time_count": len(CURRENT_NUMERIC_TIME_FEATURES_V1),
    "categorical_count": len(CURRENT_CATEGORICAL_FEATURES_V1),
    "binary_count": len(CURRENT_BINARY_FEATURES_V1),
    "user_history_count": len(USER_HISTORY_FEATURES_V1),
    "feature_set_v1_core_count": len(FEATURE_SET_V1),
    "feature_set_v1_with_derived_flags_count": len(FEATURE_SET_V1_WITH_DERIVED_FLAGS),
}, indent=2, default=str))

{
  "numeric_time_count": 9,
  "categorical_count": 4,
  "binary_count": 3,
  "user_history_count": 9,
  "feature_set_v1_core_count": 25,
  "feature_set_v1_with_derived_flags_count": 27
}


Sau muc 4, current-session features va user-history features da co checkpoint rieng. Stage tiep theo se join hai bang nay voi label/context tu modeling base de tao final `train/test_user_session_features_30d`.

## 5. Join final feature table V1

Stage nay join cac checkpoint da materialize:

- `train/test_modeling_base_v1`: key, context, label/debug label.
- `train/test_current_session_features_v1`: current-session features.
- `train/test_user_history_features_v1`: user-history features va 2 derived flags.

Final feature table giu label/debug label de validation va modeling notebook tach target, nhung khong giu cac leakage source signal nhu `session_revenue_signal` hay `session_has_purchase_signal`.

In [32]:
FINAL_CONTEXT_COLUMNS_V1 = unique_preserve_order(
    KEY_COLUMNS
    + [
        "session_date",
        "visit_start_timestamp",
        "session_year",
        "session_month",
        "has_full_30d_label_window",
    ]
)

FINAL_LABEL_COLUMNS_V1 = unique_preserve_order(LABEL_COLUMNS + LABEL_DEBUG_COLUMNS)
FINAL_FEATURE_COLUMNS_V1 = FEATURE_SET_V1_WITH_DERIVED_FLAGS

FINAL_OUTPUT_COLUMNS_V1 = unique_preserve_order(
    FINAL_CONTEXT_COLUMNS_V1
    + FINAL_LABEL_COLUMNS_V1
    + FINAL_FEATURE_COLUMNS_V1
)

FINAL_CURRENT_JOIN_FEATURES_V1 = [
    column for column in CURRENT_SESSION_FEATURES_V1
    if column not in FINAL_CONTEXT_COLUMNS_V1
]
FINAL_HISTORY_JOIN_FEATURES_V1 = USER_HISTORY_FEATURES_V1 + USER_HISTORY_DERIVED_FLAG_FEATURES_V1

print("Final context columns:", len(FINAL_CONTEXT_COLUMNS_V1))
print("Final label/debug columns:", len(FINAL_LABEL_COLUMNS_V1))
print("Final feature columns for modeling:", len(FINAL_FEATURE_COLUMNS_V1))
print("Final output columns:", len(FINAL_OUTPUT_COLUMNS_V1))
print("Current features joined from current checkpoint:", len(FINAL_CURRENT_JOIN_FEATURES_V1))
print("History features joined from history checkpoint:", len(FINAL_HISTORY_JOIN_FEATURES_V1))

Final context columns: 7
Final label/debug columns: 5
Final feature columns for modeling: 27
Final output columns: 38
Current features joined from current checkpoint: 15
History features joined from history checkpoint: 11


In [33]:
def build_final_feature_table_v1(modeling_base_path, current_feature_path, history_feature_path, output_path, dataset_name):
    print(f"Building final {dataset_name} feature table V1")

    base_df = spark.read.parquet(str(modeling_base_path)).select(
        unique_preserve_order(FINAL_CONTEXT_COLUMNS_V1 + FINAL_LABEL_COLUMNS_V1)
    )
    current_df = spark.read.parquet(str(current_feature_path)).select(
        unique_preserve_order(KEY_COLUMNS + FINAL_CURRENT_JOIN_FEATURES_V1)
    )
    history_df = spark.read.parquet(str(history_feature_path)).select(
        unique_preserve_order(KEY_COLUMNS + FINAL_HISTORY_JOIN_FEATURES_V1)
    )

    final_df = (
        base_df
        .join(current_df, on=KEY_COLUMNS, how="left")
        .join(history_df, on=KEY_COLUMNS, how="left")
        .select(FINAL_OUTPUT_COLUMNS_V1)
    )

    reset_output_path(output_path)
    (
        final_df.write
        .mode(WRITE_MODE)
        .option("maxRecordsPerFile", MAX_RECORDS_PER_FILE)
        .partitionBy(*PARTITION_COLUMNS)
        .parquet(str(output_path))
    )

    print(f"Wrote final {dataset_name} feature table V1 to {output_path}")
    return spark.read.parquet(str(output_path))


train_user_session_features_30d_df = build_final_feature_table_v1(
    TRAIN_MODELING_BASE_V1_PATH,
    TRAIN_CURRENT_SESSION_FEATURES_V1_PATH,
    TRAIN_USER_HISTORY_FEATURES_V1_PATH,
    TRAIN_FEATURE_OUTPUT_PATH,
    "train",
)

test_user_session_features_30d_df = build_final_feature_table_v1(
    TEST_MODELING_BASE_V1_PATH,
    TEST_CURRENT_SESSION_FEATURES_V1_PATH,
    TEST_USER_HISTORY_FEATURES_V1_PATH,
    TEST_FEATURE_OUTPUT_PATH,
    "test",
)

Building final train feature table V1
Wrote final train feature table V1 to G:\ds\data_pyspark_parquet\train_user_session_features_30d
Building final test feature table V1
Wrote final test feature table V1 to G:\ds\data_pyspark_parquet\test_user_session_features_30d


## 5.1 Validate final feature table V1

In [34]:
def summarize_final_feature_table_v1(df, dataset_name):
    df = df.persist(StorageLevel.DISK_ONLY)
    total_rows = df.count()
    distinct_keys = df.select(*KEY_COLUMNS).distinct().count()

    feature_null_counts = df.agg(*[
        F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
        for column in FINAL_FEATURE_COLUMNS_V1
    ]).collect()[0].asDict()

    label_row = df.agg(
        F.sum(F.when(F.col("has_full_30d_label_window").isNull(), 1).otherwise(0)).alias("null_full_window_flag_rows"),
        F.sum(F.when((F.col("has_full_30d_label_window") == 1) & F.col("future_30d_has_purchase").isNull(), 1).otherwise(0)).alias("null_class_label_full_window_rows"),
        F.sum(F.when((F.col("has_full_30d_label_window") == 1) & F.col("future_30d_revenue").isNull(), 1).otherwise(0)).alias("null_revenue_label_full_window_rows"),
        F.sum(F.when(F.col("has_full_30d_label_window") == 1, 1).otherwise(0)).alias("full_window_rows"),
        F.sum(F.when(F.col("future_30d_has_purchase") == 1, 1).otherwise(0)).alias("future_purchase_rows"),
    ).collect()[0].asDict()

    df.unpersist()
    return {
        "dataset": dataset_name,
        "row_count": total_rows,
        "distinct_key_count": distinct_keys,
        "duplicate_key_rows": total_rows - distinct_keys,
        "feature_column_count": len(FINAL_FEATURE_COLUMNS_V1),
        "output_column_count": len(df.columns),
        "feature_null_counts": feature_null_counts,
        "label_validation": label_row,
    }


train_user_session_features_30d_df = spark.read.parquet(str(TRAIN_FEATURE_OUTPUT_PATH))
test_user_session_features_30d_df = spark.read.parquet(str(TEST_FEATURE_OUTPUT_PATH))

final_schema_match = train_user_session_features_30d_df.schema == test_user_session_features_30d_df.schema
train_final_summary_v1 = summarize_final_feature_table_v1(train_user_session_features_30d_df, "train")
test_final_summary_v1 = summarize_final_feature_table_v1(test_user_session_features_30d_df, "test")

leakage_columns_in_final_output = sorted(set(LEAKAGE_EXCLUDED_COLUMNS).intersection(train_user_session_features_30d_df.columns))

print("Train final feature summary V1:")
print(json.dumps(train_final_summary_v1, indent=2, default=str))
print("Test final feature summary V1:")
print(json.dumps(test_final_summary_v1, indent=2, default=str))
print("Train/test final schema match:", final_schema_match)
print("Leakage columns in final output:", leakage_columns_in_final_output)

if not final_schema_match:
    raise ValueError("Train/test final feature schema khong khop")
if train_final_summary_v1["duplicate_key_rows"] != 0 or test_final_summary_v1["duplicate_key_rows"] != 0:
    raise ValueError("Final feature table co duplicate key fullVisitorId + visit_id")
if any(value != 0 for value in train_final_summary_v1["feature_null_counts"].values()):
    raise ValueError("Train final feature table van con null trong feature V1")
if any(value != 0 for value in test_final_summary_v1["feature_null_counts"].values()):
    raise ValueError("Test final feature table van con null trong feature V1")
if leakage_columns_in_final_output:
    raise ValueError(f"Final output dang chua leakage columns: {leakage_columns_in_final_output}")
if train_final_summary_v1["label_validation"]["null_full_window_flag_rows"] != 0 or test_final_summary_v1["label_validation"]["null_full_window_flag_rows"] != 0:
    raise ValueError("Final output co null has_full_30d_label_window")
if train_final_summary_v1["label_validation"]["null_class_label_full_window_rows"] != 0 or test_final_summary_v1["label_validation"]["null_class_label_full_window_rows"] != 0:
    raise ValueError("Final output co null classification label trong full 30d window")
if train_final_summary_v1["label_validation"]["null_revenue_label_full_window_rows"] != 0 or test_final_summary_v1["label_validation"]["null_revenue_label_full_window_rows"] != 0:
    raise ValueError("Final output co null revenue label trong full 30d window")

Train final feature summary V1:
{
  "dataset": "train",
  "row_count": 1706613,
  "distinct_key_count": 1706613,
  "duplicate_key_rows": 0,
  "feature_column_count": 27,
  "output_column_count": 38,
  "feature_null_counts": {
    "visit_number": 0,
    "totals_hits": 0,
    "totals_pageviews": 0,
    "totals_time_on_site": 0,
    "totals_new_visits": 0,
    "is_bounce": 0,
    "session_hour": 0,
    "session_day_of_week": 0,
    "session_month": 0,
    "channelGrouping_model": 0,
    "traffic_channel_type_model": 0,
    "device_category_model": 0,
    "geo_country_model": 0,
    "has_gclid": 0,
    "has_traffic_campaign": 0,
    "has_referral_path": 0,
    "user_previous_sessions": 0,
    "user_days_since_first_session": 0,
    "user_days_since_previous_session": 0,
    "user_previous_avg_pageviews": 0,
    "user_previous_avg_time_on_site": 0,
    "user_previous_bounce_rate": 0,
    "user_previous_purchase_count": 0,
    "user_previous_total_revenue": 0,
    "user_days_since_previous_p

## 6. Validation feature table

Section nay validate day du truoc khi sang modeling:

- Dataset/schema: row count, schema train/test, key uniqueness.
- Label: distribution, revenue summary, label null voi full 30d window.
- Feature: null count, numeric min/max, categorical missing/unknown rate, binary 0/1.
- Leakage: khong co leakage columns trong feature list va final output.
- Window/history: cac rule history quan trong pass.

In [35]:
FEATURE_VALIDATION_REPORT_PATH = PARQUET_DIR / "feature_validation_report"
FEATURE_VALIDATION_REPORT_JSON_PATH = PARQUET_DIR / "feature_validation_report.json"

FEATURE_COLUMNS_FOR_MODELING_V1 = FEATURE_SET_V1_WITH_DERIVED_FLAGS

NUMERIC_FEATURES_FOR_VALIDATION_V1 = [
    column for column in FEATURE_COLUMNS_FOR_MODELING_V1
    if column not in CURRENT_CATEGORICAL_FEATURES_V1
    and column not in CURRENT_BINARY_FEATURES_V1
    and column not in ["is_bounce", "is_first_session", "has_previous_purchase"]
]

BINARY_FEATURES_FOR_VALIDATION_V1 = unique_preserve_order(
    CURRENT_BINARY_FEATURES_V1
    + ["is_bounce", "is_first_session", "has_previous_purchase"]
)

CATEGORICAL_FEATURES_FOR_VALIDATION_V1 = CURRENT_CATEGORICAL_FEATURES_V1

print("Modeling feature count V1:", len(FEATURE_COLUMNS_FOR_MODELING_V1))
print("Numeric validation feature count:", len(NUMERIC_FEATURES_FOR_VALIDATION_V1))
print("Binary validation feature count:", len(BINARY_FEATURES_FOR_VALIDATION_V1))
print("Categorical validation feature count:", len(CATEGORICAL_FEATURES_FOR_VALIDATION_V1))

Modeling feature count V1: 27
Numeric validation feature count: 17
Binary validation feature count: 6
Categorical validation feature count: 4


In [36]:
def build_dataset_schema_validation(df, dataset_name):
    total_rows = df.count()
    distinct_keys = df.select(*KEY_COLUMNS).distinct().count()
    null_key_rows = df.agg(
        F.sum(F.when(F.col("fullVisitorId").isNull() | F.col("visit_id").isNull(), 1).otherwise(0)).alias("null_key_rows")
    ).collect()[0]["null_key_rows"]
    return {
        "dataset": dataset_name,
        "row_count": total_rows,
        "distinct_key_count": distinct_keys,
        "duplicate_key_rows": total_rows - distinct_keys,
        "null_key_rows": null_key_rows,
        "column_count": len(df.columns),
    }


def build_label_validation(df, dataset_name):
    labeled_df = df.where(F.col("has_full_30d_label_window") == 1)

    label_distribution_rows = (
        labeled_df
        .groupBy("future_30d_has_purchase")
        .agg(F.count(F.lit(1)).alias("row_count"))
        .orderBy("future_30d_has_purchase")
        .collect()
    )
    label_distribution = [row.asDict() for row in label_distribution_rows]

    revenue_summary = labeled_df.agg(
        F.count(F.lit(1)).alias("labeled_row_count"),
        F.sum(F.when(F.col("future_30d_has_purchase").isNull(), 1).otherwise(0)).alias("null_class_label_rows"),
        F.sum(F.when(F.col("future_30d_revenue").isNull(), 1).otherwise(0)).alias("null_revenue_label_rows"),
        F.sum(F.when(F.col("future_30d_revenue") > 0, 1).otherwise(0)).alias("positive_revenue_rows"),
        F.min("future_30d_revenue").alias("revenue_min"),
        F.expr("percentile_approx(future_30d_revenue, 0.5, 10000)").alias("revenue_p50"),
        F.expr("percentile_approx(future_30d_revenue, 0.9, 10000)").alias("revenue_p90"),
        F.expr("percentile_approx(future_30d_revenue, 0.99, 10000)").alias("revenue_p99"),
        F.avg("future_30d_revenue").alias("revenue_avg"),
        F.max("future_30d_revenue").alias("revenue_max"),
    ).collect()[0].asDict()

    total_labeled = revenue_summary["labeled_row_count"] or 0
    for item in label_distribution:
        item["row_pct"] = float(item["row_count"] / total_labeled * 100) if total_labeled else None

    return {
        "dataset": dataset_name,
        "label_distribution": label_distribution,
        "revenue_summary": revenue_summary,
    }


def build_feature_null_validation(df, dataset_name):
    null_counts = df.agg(*[
        F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
        for column in FEATURE_COLUMNS_FOR_MODELING_V1
    ]).collect()[0].asDict()
    return {"dataset": dataset_name, "feature_null_counts": null_counts}


def build_numeric_min_max_validation(df, dataset_name):
    agg_exprs = []
    for column in NUMERIC_FEATURES_FOR_VALIDATION_V1:
        agg_exprs.extend([
            F.min(column).alias(f"{column}__min"),
            F.max(column).alias(f"{column}__max"),
        ])
    row = df.agg(*agg_exprs).collect()[0].asDict()
    numeric_ranges = {
        column: {"min": row[f"{column}__min"], "max": row[f"{column}__max"]}
        for column in NUMERIC_FEATURES_FOR_VALIDATION_V1
    }
    return {"dataset": dataset_name, "numeric_min_max": numeric_ranges}


def build_categorical_validation(df, dataset_name):
    total_rows = df.count()
    agg_exprs = []
    for column in CATEGORICAL_FEATURES_FOR_VALIDATION_V1:
        agg_exprs.extend([
            F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(f"{column}__null"),
            F.sum(F.when(F.col(column) == "(missing)", 1).otherwise(0)).alias(f"{column}__missing"),
            F.sum(F.when(F.col(column) == "(unknown)", 1).otherwise(0)).alias(f"{column}__unknown"),
            F.countDistinct(column).alias(f"{column}__distinct"),
        ])
    row = df.agg(*agg_exprs).collect()[0].asDict()
    categorical_summary = {}
    for column in CATEGORICAL_FEATURES_FOR_VALIDATION_V1:
        missing_count = row[f"{column}__missing"]
        unknown_count = row[f"{column}__unknown"]
        categorical_summary[column] = {
            "null_count": row[f"{column}__null"],
            "missing_count": missing_count,
            "missing_pct": float(missing_count / total_rows * 100) if total_rows else None,
            "unknown_count": unknown_count,
            "unknown_pct": float(unknown_count / total_rows * 100) if total_rows else None,
            "distinct_count": row[f"{column}__distinct"],
        }
    return {"dataset": dataset_name, "categorical_summary": categorical_summary}


def build_binary_validation(df, dataset_name):
    row = df.agg(*[
        F.sum(F.when(~F.col(column).isin(0, 1), 1).otherwise(0)).alias(column)
        for column in BINARY_FEATURES_FOR_VALIDATION_V1
    ]).collect()[0].asDict()
    return {"dataset": dataset_name, "binary_invalid_counts": row}


def build_history_validation(df, dataset_name):
    row = df.agg(
        F.sum(F.when(F.col("user_previous_sessions") < 0, 1).otherwise(0)).alias("negative_previous_sessions"),
        F.sum(F.when((F.col("is_first_session") == 1) & (F.col("user_previous_sessions") != 0), 1).otherwise(0)).alias("first_session_with_previous_sessions"),
        F.sum(F.when((F.col("has_previous_purchase") == 1) & (F.col("user_previous_purchase_count") <= 0), 1).otherwise(0)).alias("has_previous_purchase_without_count"),
        F.sum(F.when((F.col("user_previous_purchase_count") == 0) & (F.col("user_days_since_previous_purchase") != -1), 1).otherwise(0)).alias("no_purchase_but_days_since_purchase_not_minus_one"),
    ).collect()[0].asDict()
    return {"dataset": dataset_name, "history_rule_validation": row}


def validate_no_leakage_columns():
    leakage_in_feature_list = sorted(set(LEAKAGE_EXCLUDED_COLUMNS).intersection(FEATURE_COLUMNS_FOR_MODELING_V1))
    leakage_in_final_output = sorted(set(LEAKAGE_EXCLUDED_COLUMNS).intersection(train_user_session_features_30d_df.columns))
    label_in_feature_list = sorted(set(LABEL_COLUMNS + LABEL_DEBUG_COLUMNS).intersection(FEATURE_COLUMNS_FOR_MODELING_V1))
    return {
        "leakage_in_feature_list": leakage_in_feature_list,
        "leakage_in_final_output": leakage_in_final_output,
        "label_in_feature_list": label_in_feature_list,
    }

In [37]:
train_user_session_features_30d_df = spark.read.parquet(str(TRAIN_FEATURE_OUTPUT_PATH))
test_user_session_features_30d_df = spark.read.parquet(str(TEST_FEATURE_OUTPUT_PATH))

validation_report_v1 = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "feature_set": "v1_with_derived_flags",
    "feature_count": len(FEATURE_COLUMNS_FOR_MODELING_V1),
    "schema_match": train_user_session_features_30d_df.schema == test_user_session_features_30d_df.schema,
    "dataset_schema": [
        build_dataset_schema_validation(train_user_session_features_30d_df, "train"),
        build_dataset_schema_validation(test_user_session_features_30d_df, "test"),
    ],
    "label_validation": [
        build_label_validation(train_user_session_features_30d_df, "train"),
        build_label_validation(test_user_session_features_30d_df, "test"),
    ],
    "feature_null_validation": [
        build_feature_null_validation(train_user_session_features_30d_df, "train"),
        build_feature_null_validation(test_user_session_features_30d_df, "test"),
    ],
    "numeric_min_max_validation": [
        build_numeric_min_max_validation(train_user_session_features_30d_df, "train"),
        build_numeric_min_max_validation(test_user_session_features_30d_df, "test"),
    ],
    "categorical_validation": [
        build_categorical_validation(train_user_session_features_30d_df, "train"),
        build_categorical_validation(test_user_session_features_30d_df, "test"),
    ],
    "binary_validation": [
        build_binary_validation(train_user_session_features_30d_df, "train"),
        build_binary_validation(test_user_session_features_30d_df, "test"),
    ],
    "history_validation": [
        build_history_validation(train_user_session_features_30d_df, "train"),
        build_history_validation(test_user_session_features_30d_df, "test"),
    ],
    "leakage_validation": validate_no_leakage_columns(),
}

print(json.dumps(validation_report_v1, indent=2, default=str)[:8000])

{
  "created_at_utc": "2026-06-11T10:12:49.447296+00:00",
  "feature_set": "v1_with_derived_flags",
  "feature_count": 27,
  "schema_match": true,
  "dataset_schema": [
    {
      "dataset": "train",
      "row_count": 1706613,
      "distinct_key_count": 1706613,
      "duplicate_key_rows": 0,
      "null_key_rows": 0,
      "column_count": 38
    },
    {
      "dataset": "test",
      "row_count": 401112,
      "distinct_key_count": 401112,
      "duplicate_key_rows": 0,
      "null_key_rows": 0,
      "column_count": 38
    }
  ],
  "label_validation": [
    {
      "dataset": "train",
      "label_distribution": [
        {
          "future_30d_has_purchase": 0,
          "row_count": 1602809,
          "row_pct": 98.69039541204302
        },
        {
          "future_30d_has_purchase": 1,
          "row_count": 21269,
          "row_pct": 1.3096045879569824
        }
      ],
      "revenue_summary": {
        "labeled_row_count": 1624078,
        "null_class_label_rows": 0,


In [38]:
def assert_task10_validation_passed(report):
    errors = []

    if not report["schema_match"]:
        errors.append("Train/test schema khong giong nhau")

    for row in report["dataset_schema"]:
        if row["duplicate_key_rows"] != 0:
            errors.append(f"{row['dataset']} co duplicate key rows: {row['duplicate_key_rows']}")
        if row["null_key_rows"] != 0:
            errors.append(f"{row['dataset']} co null key rows: {row['null_key_rows']}")

    for row in report["label_validation"]:
        revenue_summary = row["revenue_summary"]
        if revenue_summary["null_class_label_rows"] != 0:
            errors.append(f"{row['dataset']} co null class label trong full window")
        if revenue_summary["null_revenue_label_rows"] != 0:
            errors.append(f"{row['dataset']} co null revenue label trong full window")

    for row in report["feature_null_validation"]:
        bad_columns = {column: value for column, value in row["feature_null_counts"].items() if value != 0}
        if bad_columns:
            errors.append(f"{row['dataset']} co feature null: {bad_columns}")

    for row in report["binary_validation"]:
        bad_columns = {column: value for column, value in row["binary_invalid_counts"].items() if value != 0}
        if bad_columns:
            errors.append(f"{row['dataset']} co binary feature ngoai 0/1: {bad_columns}")

    for row in report["history_validation"]:
        bad_rules = {name: value for name, value in row["history_rule_validation"].items() if value != 0}
        if bad_rules:
            errors.append(f"{row['dataset']} history validation fail: {bad_rules}")

    leakage_validation = report["leakage_validation"]
    for key, values in leakage_validation.items():
        if values:
            errors.append(f"{key}: {values}")

    if errors:
        raise ValueError("\n".join(errors))

    return "Task 10 validation passed"


task10_validation_status = assert_task10_validation_passed(validation_report_v1)
print(task10_validation_status)

with FEATURE_VALIDATION_REPORT_JSON_PATH.open("w", encoding="utf-8") as file:
    json.dump(validation_report_v1, file, ensure_ascii=False, indent=2, default=str)

print("Wrote validation report to:", FEATURE_VALIDATION_REPORT_JSON_PATH)

Task 10 validation passed
Wrote validation report to: G:\ds\data_pyspark_parquet\feature_validation_report.json
